# FlightRisk VN — Thành viên 8: Web Analytics Dashboard (Part 6C)

Notebook kiểm tra kết nối API và ba kịch bản UAT của giao diện TV7/TV8. Kết quả lấy từ máy chủ và bộ artifact hiện hành; không dùng kết quả giả lập. Xem biên bản `docs/REPORT_MEMBER_8.md`.

## 1. Xác nhận thư mục dự án
Có thể chạy notebook từ thư mục gốc hoặc thư mục `notebooks/`.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'config.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Hãy chạy notebook từ thư mục gốc dự án hoặc notebooks/.')
sys.path.insert(0, str(ROOT))
for path in ('web/static/app.js', 'web/templates/index.html', 'tests/test_web_dashboard.py'):
    assert (ROOT / path).is_file(), f'Thiếu {path}'
print('Dự án:', ROOT)

## 2. Kết nối API
Chạy `.\venv\Scripts\python.exe main.py web` tại thư mục gốc trước khi thực hiện các ô dưới đây. Nếu `/health` báo lỗi, kiểm tra artifact và môi trường theo báo cáo TV6.

In [ ]:
import json
from urllib.request import Request, urlopen

BASE_URL = 'http://127.0.0.1:5000'
def api(path, payload=None):
    body = None if payload is None else json.dumps(payload).encode('utf-8')
    headers = {} if body is None else {'Content-Type': 'application/json'}
    with urlopen(Request(BASE_URL + path, data=body, headers=headers), timeout=30) as response:
        return json.load(response)

health = api('/health')
print('Health:', health)
assert health.get('ready') is True, 'Artifact chưa sẵn sàng'
stats, clusters, rules = api('/api/stats'), api('/api/clusters'), api('/api/rules')
print('Mô hình được chọn:', stats['supervised']['model_selection']['selected'])
print('Số cụm:', len(clusters['profiles']))
print('Số luật API trả về:', len(rules), '; dashboard hiển thị tối đa 15')

## 3. Kịch bản UAT với API thực
Các tình huống khớp preset trên giao diện. Kết quả được in trực tiếp từ API, có thể khác kết quả mong đợi trong hướng dẫn; không sửa số liệu để ép khớp.

In [ ]:
common = {'direction': 'DEPARTURES', 'route_type': 'domestic', 'flight_date': '2026-11-06'}
cases = {
    'safe': dict(airline_code='VN', airport='SGN', origin='SGN', destination='HAN', scheduled_time='08:00', precipitation=0, wind_gusts_10m=10, visibility=10000, weather_code=0, airport_load=15),
    'windy': dict(airline_code='QH', airport='DAD', origin='DAD', destination='SGN', scheduled_time='14:45', precipitation=1, wind_gusts_10m=32, visibility=6000, weather_code=61, airport_load=26),
    'storm': dict(airline_code='VJ', airport='SGN', origin='SGN', destination='HAN', scheduled_time='19:30', precipitation=15, wind_gusts_10m=45, visibility=2500, weather_code=95, airport_load=35),
}
required = {'prediction', 'risk_score', 'risk_level', 'estimated_delay_min', 'probabilities', 'cluster_profile', 'matched_rules'}
for name, params in cases.items():
    result = api('/api/predict', common | params)
    assert required.issubset(result), f'Thiếu trường trong kết quả {name}'
    probabilities = result['probabilities']
    assert abs(sum(probabilities.values()) - 1) < 0.001
    print(name, json.dumps({key: result[key] for key in ('prediction', 'risk_score', 'risk_level', 'estimated_delay_min', 'probabilities')}, ensure_ascii=False))

## 4. Kiểm thử trình duyệt
Cần Playwright và Chrome/Chromium/Edge. Nếu thiếu, các bài liên quan được báo `skipped`, không phải đã kiểm thử thành công.

In [ ]:
import unittest
suite = unittest.defaultTestLoader.loadTestsFromNames([
    'tests.test_web_dashboard', 'tests.test_web_frontend_browser'
])
result = unittest.TextTestRunner(verbosity=2).run(suite)
print('Số bài bỏ qua:', len(result.skipped))
assert result.wasSuccessful(), 'Có kiểm thử thất bại'

## 5. Diễn giải
So sánh kết quả chạy thực tế với `docs/huong_dan_thanh_vien.md` và ghi lại những khác biệt trong biên bản. `risk_score` là tổng xác suất trễ và hủy; lớp dự báo là lớp có xác suất riêng cao nhất. Độ trễ dưới 50 ms cần benchmark riêng, không suy ra từ việc giao diện hoạt động.